## Tech Challenge — Fase 4  
### Notebook 01 — Análise de Vídeo com YOLOv8 Pose

Este notebook faz parte da Fase 4 do Tech Challenge da Pós-graduação em Inteligência Artificial para Devs.

O objetivo desta etapa é validar a modalidade de **vídeo** dentro de uma solução multimodal voltada ao apoio preventivo em saúde da mulher. A proposta é utilizar visão computacional para identificar possíveis sinais comportamentais de desconforto, medo ou vulnerabilidade durante uma consulta simulada.

A abordagem adotada segue as orientações recebidas na disciplina:

- utilizar fontes abertas de dados;
- utilizar modelos pré-treinados para acelerar o desenvolvimento;
- escolher modalidades específicas em vez de tentar resolver todos os cenários possíveis;
- demonstrar uma solução funcional, acadêmica e tecnicamente justificável.

Neste notebook, a modalidade escolhida é **vídeo**, utilizando **YOLOv8 Pose** para detecção de pessoas e pontos corporais.

### 1. Estratégia Técnica Adotada

A estratégia escolhida para este notebook é utilizar o modelo **YOLOv8 Pose**, da biblioteca `ultralytics`, em vez de treinar um detector médico do zero.

Essa decisão foi tomada por quatro motivos principais:

1. o professor confirmou que modelos pré-treinados podem e devem ser utilizados para acelerar o desenvolvimento;
2. o desafio não fornece datasets médicos específicos;
3. a atividade espera pesquisa e uso de fontes abertas como Kaggle, Hugging Face e outras bases públicas;
4. o objetivo acadêmico principal é demonstrar a integração e a aplicação prática de IA multimodal.

O YOLOv8 Pose permite detectar pessoas e estimar pontos corporais, como ombros, braços, mãos, quadril e cabeça. A partir desses pontos, o projeto aplica uma camada de regras heurísticas para estimar sinais comportamentais relevantes.

Essa camada heurística representa a adaptação do modelo para o contexto do projeto.

### 2. Instalação das Dependências

Esta célula instala as bibliotecas necessárias para execução do notebook.

Principais bibliotecas utilizadas:

- `ultralytics`: execução do modelo YOLOv8;
- `opencv-python`: leitura e manipulação de vídeos;
- `matplotlib`: visualização de frames;
- `pandas`: organização dos resultados;
- `numpy`: cálculos auxiliares;
- `tqdm`: barra de progresso.

### 3. Importação das Bibliotecas

Nesta etapa são importadas as bibliotecas utilizadas no pipeline de análise visual.

O notebook foi estruturado para funcionar tanto com vídeos reais quanto com vídeos simulados. Caso nenhum vídeo seja informado, será possível testar o fluxo com um vídeo de exemplo informado pelo usuário posteriormente.

In [ ]:
import os
import json
import math
from pathlib import Path
from datetime import datetime

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

from ultralytics import YOLO

### 4. Configuração dos Diretórios

Nesta etapa são definidos os caminhos utilizados pelo projeto.

A estrutura segue a organização proposta para a Fase 4 dentro do repositório existente:

- `dados/Fase 4/video`: vídeos de entrada;
- `resultados/Fase 4/video`: frames anotados e resultados da análise visual;
- `resultados/Fase 4/reports`: relatórios técnicos gerados automaticamente.

Essa separação facilita a execução pelo professor e mantém consistência com as fases anteriores do projeto.

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

VIDEO_DIR = PROJECT_ROOT / "dados" / "Fase 4" / "video"
VIDEO_RESULTS_DIR = PROJECT_ROOT / "resultados" / "Fase 4" / "video"
REPORTS_DIR = PROJECT_ROOT / "resultados" / "Fase 4" / "reports"

for directory in [VIDEO_DIR, VIDEO_RESULTS_DIR, REPORTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print("Diretório de vídeos:", VIDEO_DIR)
print("Diretório de resultados de vídeo:", VIDEO_RESULTS_DIR)
print("Diretório de relatórios:", REPORTS_DIR)

### 5. Definição do Vídeo de Entrada

O vídeo de entrada deve representar uma consulta, entrevista ou interação simulada em que seja possível observar postura corporal e sinais não verbais.

Como o desafio não fornece datasets oficiais, recomenda-se utilizar fontes abertas, como:

- Kaggle;
- Hugging Face;
- GitHub acadêmico;
- datasets públicos de comportamento, emoção ou violência;
- vídeos simulados produzidos pelo próprio grupo.

Para manter o projeto eticamente adequado, o vídeo deve ser tratado como **simulação acadêmica**, não como evidência clínica real.

Coloque o arquivo de vídeo dentro de:

`dados/Fase 4/video/`

Depois informe o nome do arquivo na variável `VIDEO_FILE_NAME`.

In [ ]:
# Altere este nome para o arquivo de vídeo usado no experimento.
# Exemplo: "sample_consulta.mp4"
VIDEO_FILE_NAME = "vio_2.mp4"

video_path = VIDEO_DIR / VIDEO_FILE_NAME

if not video_path.exists():
    print(f"Vídeo não encontrado em: {video_path}")
    print("Coloque um arquivo .mp4 em dados/Fase 4/video/ ou altere VIDEO_FILE_NAME.")
else:
    print(f"Vídeo encontrado: {video_path}")

### 6. Leitura das Informações do Vídeo

Antes de executar a análise com IA, é importante entender as características do vídeo:

- resolução;
- quantidade de frames;
- taxa de quadros por segundo;
- duração aproximada.

Essas informações ajudam a definir a frequência de amostragem dos frames e o custo computacional da análise.

In [ ]:
def get_video_metadata(path: Path) -> dict:
    """Retorna metadados básicos de um arquivo de vídeo."""
    cap = cv2.VideoCapture(str(path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir o vídeo: {path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    duration = frame_count / fps if fps else 0
    cap.release()

    return {
        "fps": fps,
        "frame_count": frame_count,
        "width": width,
        "height": height,
        "duration_seconds": duration,
    }

if video_path.exists():
    metadata = get_video_metadata(video_path)
    display(pd.DataFrame([metadata]))
else:
    metadata = None

### 7. Extração de Frames

A análise de vídeo pode ser computacionalmente custosa quando todos os frames são processados.

Por isso, este notebook utiliza uma estratégia de amostragem: em vez de analisar todos os frames, são extraídos frames em intervalos regulares.

Essa abordagem é adequada para um protótipo acadêmico porque:

- reduz o tempo de processamento;
- mantém evidências visuais suficientes para a demonstração;
- facilita a inspeção dos resultados;
- permite execução em computadores comuns.

In [ ]:
def extract_sampled_frames(video_path: Path, output_dir: Path, sample_every_seconds: float = 1.0, max_frames: int = 30):
    """
    Extrai frames amostrados de um vídeo.

    Args:
        video_path: caminho do vídeo.
        output_dir: diretório onde os frames serão salvos.
        sample_every_seconds: intervalo de amostragem em segundos.
        max_frames: quantidade máxima de frames extraídos.

    Returns:
        Lista com caminhos dos frames extraídos.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        raise FileNotFoundError(f"Não foi possível abrir o vídeo: {video_path}")

    fps = cap.get(cv2.CAP_PROP_FPS)
    sample_interval = max(int(fps * sample_every_seconds), 1)

    frame_paths = []
    frame_index = 0
    saved_count = 0

    while cap.isOpened() and saved_count < max_frames:
        ret, frame = cap.read()
        if not ret:
            break

        if frame_index % sample_interval == 0:
            frame_name = f"frame_{saved_count:04d}.jpg"
            frame_path = output_dir / frame_name
            cv2.imwrite(str(frame_path), frame)
            frame_paths.append(frame_path)
            saved_count += 1

        frame_index += 1

    cap.release()
    return frame_paths

frames_dir = VIDEO_RESULTS_DIR / "sampled_frames"

if video_path.exists():
    frame_paths = extract_sampled_frames(video_path, frames_dir, sample_every_seconds=1.0, max_frames=30)
    print(f"Frames extraídos: {len(frame_paths)}")
else:
    frame_paths = []

### 8. Visualização Inicial dos Frames

A visualização dos frames extraídos permite validar se o vídeo foi carregado corretamente e se os frames representam cenas úteis para análise comportamental.

Esta etapa também é importante para documentação, pois permite incluir evidências visuais no relatório técnico do projeto.

In [ ]:
def show_frames(frame_paths, max_images=6):
    if not frame_paths:
        print("Nenhum frame disponível para visualização.")
        return

    selected = frame_paths[:max_images]
    plt.figure(figsize=(15, 8))

    for idx, path in enumerate(selected, start=1):
        image = cv2.imread(str(path))
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        plt.subplot(2, 3, idx)
        plt.imshow(image)
        plt.title(path.name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

show_frames(frame_paths)

### 9. Carregamento do Modelo YOLOv8 Pose

Nesta etapa é carregado o modelo `yolov8n-pose.pt`, uma versão leve do YOLOv8 especializada em estimativa de pose humana.

O modelo identifica pessoas e estima pontos corporais, como:

- cabeça;
- ombros;
- cotovelos;
- punhos;
- quadril;
- joelhos;
- tornozelos.

A versão `n` foi escolhida por ser mais leve e adequada para execução acadêmica em máquinas comuns.

Caso exista GPU disponível, a inferência será acelerada automaticamente pela biblioteca.

In [ ]:
model = YOLO("yolov8n-pose.pt")
print("Modelo YOLOv8 Pose carregado com sucesso.")

### 10. Inferência Visual com YOLOv8 Pose

Nesta etapa o modelo é aplicado aos frames extraídos.

Para cada frame, o pipeline coleta:

- quantidade de pessoas detectadas;
- confiança da detecção;
- keypoints corporais;
- frame anotado com skeleton/pose;
- evidências utilizadas na geração de score.

Esses dados serão usados posteriormente na etapa de interpretação comportamental.

In [ ]:
def run_pose_inference(model, frame_paths, output_dir: Path, confidence_threshold: float = 0.25):
    """Executa YOLOv8 Pose nos frames e salva imagens anotadas."""
    annotated_dir = output_dir / "annotated_frames"
    annotated_dir.mkdir(parents=True, exist_ok=True)

    results_data = []

    for frame_path in tqdm(frame_paths, desc="Processando frames"):
        results = model.predict(str(frame_path), conf=confidence_threshold, verbose=False)
        result = results[0]

        annotated = result.plot()
        annotated_path = annotated_dir / frame_path.name
        cv2.imwrite(str(annotated_path), annotated)

        boxes = result.boxes
        keypoints = result.keypoints

        person_count = len(boxes) if boxes is not None else 0
        confidences = boxes.conf.cpu().numpy().tolist() if boxes is not None and boxes.conf is not None else []

        keypoints_xy = None
        keypoints_conf = None

        if keypoints is not None and keypoints.xy is not None:
            keypoints_xy = keypoints.xy.cpu().numpy().tolist()
        if keypoints is not None and keypoints.conf is not None:
            keypoints_conf = keypoints.conf.cpu().numpy().tolist()

        results_data.append({
            "frame": frame_path.name,
            "frame_path": str(frame_path),
            "annotated_path": str(annotated_path),
            "person_count": person_count,
            "confidences": confidences,
            "keypoints_xy": keypoints_xy,
            "keypoints_conf": keypoints_conf,
        })

    return results_data

if frame_paths:
    pose_results = run_pose_inference(model, frame_paths, VIDEO_RESULTS_DIR)
    print(f"Frames analisados: {len(pose_results)}")
else:
    pose_results = []

### 11. Visualização dos Frames Anotados

Os frames anotados permitem verificar visualmente se o YOLOv8 conseguiu identificar pessoas e estimar a pose corporal.

Essa etapa é importante porque fornece evidências qualitativas para o relatório e para o vídeo demonstrativo da entrega.

In [ ]:
annotated_paths = [Path(item["annotated_path"]) for item in pose_results]
show_frames(annotated_paths)

### 12. Camada Heurística de Interpretação Comportamental

O YOLOv8 Pose identifica pontos corporais, mas não interpreta diretamente sentimentos, medo ou desconforto.

Por isso, foi criada uma camada heurística simples para transformar evidências visuais em indicadores comportamentais.

As heurísticas abaixo são aproximações acadêmicas, não diagnósticos clínicos:

1. **mãos próximas ao rosto**: pode indicar tensão, choro, proteção ou desconforto;
2. **braços próximos ao tronco**: pode indicar postura defensiva ou retraída;
3. **movimento corporal elevado entre frames**: pode indicar agitação;
4. **ausência ou baixa confiança de detecção**: reduz a confiabilidade da análise.

A pontuação visual final é calculada com base na presença desses sinais ao longo dos frames analisados.

In [ ]:
COCO_KEYPOINTS = {
    "nose": 0,
    "left_shoulder": 5,
    "right_shoulder": 6,
    "left_elbow": 7,
    "right_elbow": 8,
    "left_wrist": 9,
    "right_wrist": 10,
    "left_hip": 11,
    "right_hip": 12,
}


def euclidean_distance(p1, p2):
    return float(np.linalg.norm(np.array(p1) - np.array(p2)))


def safe_point(person_keypoints, idx):
    try:
        point = person_keypoints[idx]
        if point[0] == 0 and point[1] == 0:
            return None
        return point
    except Exception:
        return None


def analyze_person_posture(person_keypoints):
    """
    Analisa uma pessoa detectada e retorna indicadores heurísticos.
    """
    nose = safe_point(person_keypoints, COCO_KEYPOINTS["nose"])
    left_wrist = safe_point(person_keypoints, COCO_KEYPOINTS["left_wrist"])
    right_wrist = safe_point(person_keypoints, COCO_KEYPOINTS["right_wrist"])
    left_shoulder = safe_point(person_keypoints, COCO_KEYPOINTS["left_shoulder"])
    right_shoulder = safe_point(person_keypoints, COCO_KEYPOINTS["right_shoulder"])
    left_hip = safe_point(person_keypoints, COCO_KEYPOINTS["left_hip"])
    right_hip = safe_point(person_keypoints, COCO_KEYPOINTS["right_hip"])

    indicators = {
        "hands_near_face": False,
        "defensive_posture": False,
        "analysis_confidence": "partial",
    }

    # Distância de referência baseada na largura dos ombros.
    shoulder_width = None
    if left_shoulder is not None and right_shoulder is not None:
        shoulder_width = euclidean_distance(left_shoulder, right_shoulder)

    if shoulder_width is None or shoulder_width == 0:
        return indicators

    # Heurística 1: mãos próximas ao rosto.
    if nose is not None:
        wrist_distances = []
        if left_wrist is not None:
            wrist_distances.append(euclidean_distance(left_wrist, nose))
        if right_wrist is not None:
            wrist_distances.append(euclidean_distance(right_wrist, nose))

        if wrist_distances and min(wrist_distances) < shoulder_width * 0.9:
            indicators["hands_near_face"] = True

    # Heurística 2: braços próximos ao centro do corpo.
    if left_wrist is not None and right_wrist is not None and left_hip is not None and right_hip is not None:
        body_center = ((left_hip[0] + right_hip[0]) / 2, (left_hip[1] + right_hip[1]) / 2)
        left_distance = euclidean_distance(left_wrist, body_center)
        right_distance = euclidean_distance(right_wrist, body_center)

        if left_distance < shoulder_width * 1.2 and right_distance < shoulder_width * 1.2:
            indicators["defensive_posture"] = True

    indicators["analysis_confidence"] = "good"
    return indicators


def analyze_frame_behavior(pose_item):
    """Analisa todos os indivíduos detectados em um frame."""
    frame_indicators = {
        "frame": pose_item["frame"],
        "person_count": pose_item["person_count"],
        "hands_near_face_count": 0,
        "defensive_posture_count": 0,
        "behavior_score": 0.0,
    }

    keypoints_xy = pose_item.get("keypoints_xy")
    if not keypoints_xy:
        return frame_indicators

    for person in keypoints_xy:
        indicators = analyze_person_posture(person)
        if indicators["hands_near_face"]:
            frame_indicators["hands_near_face_count"] += 1
        if indicators["defensive_posture"]:
            frame_indicators["defensive_posture_count"] += 1

    score = 0.0
    if frame_indicators["hands_near_face_count"] > 0:
        score += 0.45
    if frame_indicators["defensive_posture_count"] > 0:
        score += 0.35
    if frame_indicators["person_count"] > 1:
        score += 0.10

    frame_indicators["behavior_score"] = min(score, 1.0)
    return frame_indicators

behavior_results = [analyze_frame_behavior(item) for item in pose_results]
behavior_df = pd.DataFrame(behavior_results)
behavior_df.head()

### 13. Cálculo do Score Visual

O score visual consolida os sinais detectados ao longo dos frames.

A lógica adotada é simples e explicável, adequada a um protótipo acadêmico:

- quanto maior a frequência de sinais visuais de desconforto, maior o score;
- frames sem pessoa detectada ou sem keypoints confiáveis têm score baixo;
- o score final representa a média dos scores dos frames analisados.

A classificação segue três níveis:

- **baixo**: poucos ou nenhum indicador visual;
- **moderado**: alguns sinais aparecem em parte dos frames;
- **alto**: sinais recorrentes em vários frames.

In [ ]:
def classify_visual_risk(score: float) -> str:
    if score >= 0.65:
        return "ALTO"
    if score >= 0.35:
        return "MODERADO"
    return "BAIXO"

if not behavior_df.empty:
    visual_score = float(behavior_df["behavior_score"].mean())
    max_frame_score = float(behavior_df["behavior_score"].max())
    risk_level = classify_visual_risk(visual_score)
else:
    visual_score = 0.0
    max_frame_score = 0.0
    risk_level = "INDEFINIDO"

summary = {
    "visual_score": round(visual_score, 4),
    "max_frame_score": round(max_frame_score, 4),
    "visual_risk_level": risk_level,
    "frames_analyzed": len(behavior_df),
    "frames_with_hands_near_face": int((behavior_df["hands_near_face_count"] > 0).sum()) if not behavior_df.empty else 0,
    "frames_with_defensive_posture": int((behavior_df["defensive_posture_count"] > 0).sum()) if not behavior_df.empty else 0,
}

summary

### 14. Visualização dos Resultados por Frame

A tabela abaixo apresenta o resultado da análise visual em cada frame.

Ela permite verificar:

- em quais frames houve detecção de pessoas;
- quando sinais heurísticos foram identificados;
- como o score visual varia ao longo do vídeo.

Essa tabela pode ser utilizada posteriormente no relatório técnico e no pipeline final de fusão multimodal.

In [ ]:
behavior_df

### 15. Gráfico do Score Visual ao Longo do Vídeo

O gráfico ajuda a identificar momentos do vídeo em que os sinais visuais foram mais intensos.

Na solução final, esses picos podem ser usados para destacar trechos relevantes para revisão da equipe médica.

In [ ]:
if not behavior_df.empty:
    plt.figure(figsize=(12, 5))
    plt.plot(behavior_df.index, behavior_df["behavior_score"], marker="o")
    plt.title("Score Visual por Frame Amostrado")
    plt.xlabel("Frame amostrado")
    plt.ylabel("Score visual")
    plt.ylim(0, 1)
    plt.grid(True)
    plt.show()
else:
    print("Não há dados para plotar.")

### 16. Geração de Relatório Automático

O documento da Fase 4 solicita relatórios automáticos especializados.

Nesta etapa é gerado um relatório JSON contendo:

- data e hora da análise;
- vídeo analisado;
- quantidade de frames processados;
- score visual;
- nível de risco visual;
- evidências encontradas;
- observação ética sobre o uso do resultado.

Esse relatório será utilizado futuramente pela etapa de fusão multimodal e pela interface de demonstração.

In [ ]:
def generate_video_report(video_file_name: str, summary: dict, behavior_df: pd.DataFrame, output_dir: Path):
    report = {
        "report_type": "video_analysis",
        "phase": "Fase 4",
        "generated_at": datetime.now().isoformat(),
        "video_file": video_file_name,
        "method": "YOLOv8 Pose pre-trained model + behavioral heuristic layer",
        "summary": summary,
        "evidence": {
            "frames": behavior_df.to_dict(orient="records") if not behavior_df.empty else [],
        },
        "ethical_note": (
            "This analysis is intended for academic demonstration and preventive decision support only. "
            "It must not be interpreted as a clinical diagnosis or definitive evidence of abuse or violence."
        ),
    }

    output_path = output_dir / "video_analysis_report.json"
    with open(output_path, "w", encoding="utf-8") as file:
        json.dump(report, file, indent=4, ensure_ascii=False)

    return output_path, report

report_path, report = generate_video_report(VIDEO_FILE_NAME, summary, behavior_df, REPORTS_DIR)
print(f"Relatório gerado em: {report_path}")

### 17. Interpretação Técnica do Resultado

A interpretação abaixo resume o comportamento visual observado no vídeo de entrada.

Ela utiliza os indicadores gerados pela camada heurística e apresenta uma leitura compatível com o objetivo acadêmico do projeto.

O resultado não deve ser interpretado como diagnóstico, mas como sinal de apoio para investigação ou revisão humana.

In [ ]:
def build_interpretation(summary: dict) -> str:
    risk_level = summary["visual_risk_level"]
    score = summary["visual_score"]

    if risk_level == "ALTO":
        return (
            f"O vídeo apresentou score visual {score:.2f}, classificado como ALTO. "
            "Foram identificados sinais visuais recorrentes que podem indicar desconforto, postura defensiva "
            "ou vulnerabilidade. Recomenda-se revisão humana especializada."
        )

    if risk_level == "MODERADO":
        return (
            f"O vídeo apresentou score visual {score:.2f}, classificado como MODERADO. "
            "Foram observados alguns sinais visuais compatíveis com desconforto ou postura defensiva. "
            "A interpretação deve ser combinada com áudio e texto na etapa multimodal."
        )

    if risk_level == "BAIXO":
        return (
            f"O vídeo apresentou score visual {score:.2f}, classificado como BAIXO. "
            "Poucos sinais visuais de desconforto foram identificados nos frames analisados. "
            "Ainda assim, a avaliação final dependerá da fusão com outras modalidades."
        )

    return "Não foi possível definir o nível de risco visual com os dados disponíveis."

interpretation = build_interpretation(summary)
print(interpretation)

### 18. Exportação dos Resultados em CSV

Além do relatório JSON, os resultados por frame são exportados em CSV.

Esse arquivo facilita:

- inspeção manual;
- geração de gráficos;
- uso posterior no pipeline de fusão multimodal;
- documentação dos experimentos.

In [ ]:
csv_path = VIDEO_RESULTS_DIR / "video_behavior_results.csv"
behavior_df.to_csv(csv_path, index=False, encoding="utf-8")
print(f"CSV exportado em: {csv_path}")

### 19. Como Este Notebook Será Integrado aos Próximos Módulos

O resultado deste notebook será utilizado nas próximas etapas da Fase 4.

Fluxo previsto:

1. este notebook gera o **score visual**;
2. o notebook de áudio gera o **score emocional/vocal**;
3. o notebook de fusão multimodal combina os scores;
4. o notebook de Azure demonstra integração com serviços em nuvem;
5. o pipeline final consolida tudo em um dashboard e relatório.

Dessa forma, este notebook representa a primeira modalidade da solução multimodal.

### 20. Limitações da Abordagem

Como este é um protótipo acadêmico, algumas limitações devem ser explicitadas:

- o modelo utilizado é pré-treinado e não foi desenvolvido especificamente para saúde da mulher;
- os sinais visuais analisados são heurísticos e não clínicos;
- datasets públicos podem não representar perfeitamente o contexto hospitalar;
- a análise depende da qualidade do vídeo, iluminação, enquadramento e visibilidade corporal;
- a saída deve ser usada apenas como apoio preventivo e não como decisão automática.

Essas limitações são importantes para demonstrar responsabilidade ética e técnica no uso de IA em saúde.

## 21. Conclusão

Este notebook demonstrou a viabilidade da análise visual com YOLOv8 Pose como parte da solução multimodal da Fase 4.

Foram executadas as seguintes etapas:

- carregamento de vídeo;
- extração de frames;
- inferência com YOLOv8 Pose;
- análise heurística de postura e sinais não verbais;
- geração de score visual;
- exportação de relatório JSON e CSV.

A abordagem está alinhada com as orientações do professor, pois utiliza modelo pré-treinado, aceita datasets abertos e foca em uma solução acadêmica funcional.

Nos próximos notebooks, este score visual será combinado com análise de áudio e texto para compor a decisão multimodal final.